# SQL Practice — HR Database
> Write your own queries in the **Your answer** cells, then run the **Answer** cells to check.

## Schema
```
employees      (emp_id, name, dept_id, salary, hire_date, manager_id)
departments    (dept_id, dept_name, location)
projects       (proj_id, proj_name, budget, dept_id)
project_assignments (emp_id, proj_id, hours)
```

## Exercise list
| # | Topic |
|---|---|
| E1 | SELECT + WHERE |
| E2 | CONCAT — full name |
| E3 | GROUP BY + COUNT |
| E4 | GROUP BY + HAVING |
| E5 | INNER JOIN |
| E6 | LEFT JOIN |
| E7 | Subquery |
| E8 | COUNT DISTINCT + HAVING |
| E9 | ORDER BY + LIMIT |
| E10 | UPDATE + SELECT |

In [ ]:
# ── Setup — run this first ──────────────────────────────────────
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')

conn.executescript("""
CREATE TABLE departments (
    dept_id   INTEGER PRIMARY KEY,
    dept_name TEXT,
    location  TEXT
);
CREATE TABLE employees (
    emp_id     INTEGER PRIMARY KEY,
    name       TEXT,
    dept_id    INTEGER,
    salary     REAL,
    hire_date  TEXT,
    manager_id INTEGER
);
CREATE TABLE projects (
    proj_id   INTEGER PRIMARY KEY,
    proj_name TEXT,
    budget    REAL,
    dept_id   INTEGER
);
CREATE TABLE project_assignments (
    emp_id  INTEGER,
    proj_id INTEGER,
    hours   INTEGER
);

INSERT INTO departments VALUES
    (1,'Engineering','New York'),
    (2,'Marketing','Chicago'),
    (3,'HR','Boston'),
    (4,'Finance','Dallas'),
    (5,'Legal','Miami');      -- no employees assigned

INSERT INTO employees VALUES
    (1, 'Alice Smith',    1, 120000, '2018-03-15', NULL),
    (2, 'Bob Jones',      1,  95000, '2019-07-01', 1),
    (3, 'Carol White',    1, 105000, '2020-01-10', 1),
    (4, 'David Brown',    2,  80000, '2017-11-20', NULL),
    (5, 'Eve Martinez',   2,  72000, '2021-05-03', 4),
    (6, 'Frank Wilson',   2,  85000, '2016-08-14', 4),
    (7, 'Grace Lee',      3,  70000, '2022-02-28', NULL),
    (8, 'Hank Thompson',  3,  68000, '2022-09-15', 7),
    (9, 'Ivy Chen',       4,  92000, '2015-06-01', NULL),
    (10,'Jack Davis',     4,  88000, '2021-12-01', 9);

INSERT INTO projects VALUES
    (1, 'Data Pipeline',   150000, 1),
    (2, 'API Redesign',    200000, 1),
    (3, 'SEO Campaign',     80000, 2),
    (4, 'Budget Review',   50000, 4);

INSERT INTO project_assignments VALUES
    (1,1,120),(1,2,80),
    (2,1,100),(2,2,60),
    (3,2,150),
    (4,3,200),
    (5,3,90),
    (9,4,40),(9,1,30),
    (10,4,60);
""")

def q(sql): return pd.read_sql_query(sql, conn)

print('✅ Database ready!')
print('Tables: employees, departments, projects, project_assignments')
print(f'  {len(q("SELECT * FROM employees"))} employees  |  {len(q("SELECT * FROM departments"))} departments')
print(f'  {len(q("SELECT * FROM projects"))} projects   |  {len(q("SELECT * FROM project_assignments"))} assignments')

---
## E1 · SELECT + WHERE
**Task:** Retrieve the name and salary of all employees earning more than **80,000**.

In [ ]:
# ── Your answer ────────────────────────────────────────────────
q("""

""")

In [ ]:
# ── Answer E1 ──────────────────────────────────────────────────
q("""
    SELECT name, salary
    FROM   employees
    WHERE  salary > 80000
    ORDER BY salary DESC
""")

---
## E2 · CONCAT — Full Name
**Task:** Show each employee's full name (first + last concatenated) as `full_name` and their `salary`.

In [ ]:
# ── Your answer ────────────────────────────────────────────────
q("""

""")

In [ ]:
# ── Answer E2 ──────────────────────────────────────────────────
# SQLite: use || for concatenation (MySQL uses CONCAT)
q("""
    SELECT name AS full_name, salary
    FROM   employees
    ORDER BY name
""")
# If name were split into first/last columns:
# SELECT first_name || ' ' || last_name AS full_name, salary FROM employees

---
## E3 · GROUP BY + COUNT
**Task:** Count how many employees are in each department. Show `dept_id` and `headcount`, sorted by headcount descending.

In [ ]:
# ── Your answer ────────────────────────────────────────────────
q("""

""")

In [ ]:
# ── Answer E3 ──────────────────────────────────────────────────
q("""
    SELECT   dept_id,
             COUNT(*) AS headcount
    FROM     employees
    GROUP BY dept_id
    ORDER BY headcount DESC
""")

---
## E4 · GROUP BY + HAVING
**Task:** Show departments where the **average salary exceeds 85,000**. Include `dept_id` and `avg_salary`.

In [ ]:
# ── Your answer ────────────────────────────────────────────────
q("""

""")

In [ ]:
# ── Answer E4 ──────────────────────────────────────────────────
q("""
    SELECT   dept_id,
             ROUND(AVG(salary), 0) AS avg_salary
    FROM     employees
    GROUP BY dept_id
    HAVING   AVG(salary) > 85000
    ORDER BY avg_salary DESC
""")

---
## E5 · INNER JOIN
**Task:** Show each employee's `name`, their `dept_name`, and `salary`. Only include employees who have a department assigned.

In [ ]:
# ── Your answer ────────────────────────────────────────────────
q("""

""")

In [ ]:
# ── Answer E5 ──────────────────────────────────────────────────
q("""
    SELECT   e.name, d.dept_name, e.salary
    FROM     employees e
    JOIN     departments d ON e.dept_id = d.dept_id
    ORDER BY d.dept_name, e.salary DESC
""")

---
## E6 · LEFT JOIN
**Task:** Show ALL departments and how many employees each has. Departments with **no employees** should show `0`.

In [ ]:
# ── Your answer ────────────────────────────────────────────────
q("""

""")

In [ ]:
# ── Answer E6 ──────────────────────────────────────────────────
q("""
    SELECT   d.dept_name,
             COUNT(e.emp_id) AS headcount
    FROM     departments d
    LEFT JOIN employees e ON d.dept_id = e.dept_id
    GROUP BY d.dept_id, d.dept_name
    ORDER BY headcount DESC
""")

---
## E7 · Subquery
**Task:** Find employees whose salary is **above the company-wide average**. Show name, salary, and the average as a column.

In [ ]:
# ── Your answer ────────────────────────────────────────────────
q("""

""")

In [ ]:
# ── Answer E7 ──────────────────────────────────────────────────
q("""
    SELECT name,
           salary,
           ROUND((SELECT AVG(salary) FROM employees), 0) AS company_avg
    FROM   employees
    WHERE  salary > (SELECT AVG(salary) FROM employees)
    ORDER BY salary DESC
""")

---
## E8 · COUNT DISTINCT + HAVING
**Task:** Find employees assigned to **more than 1 project**. Show their `name` and `project_count`.

In [ ]:
# ── Your answer ────────────────────────────────────────────────
q("""

""")

In [ ]:
# ── Answer E8 ──────────────────────────────────────────────────
q("""
    SELECT   e.name,
             COUNT(DISTINCT pa.proj_id) AS project_count
    FROM     employees e
    JOIN     project_assignments pa ON e.emp_id = pa.emp_id
    GROUP BY e.emp_id, e.name
    HAVING   COUNT(DISTINCT pa.proj_id) > 1
    ORDER BY project_count DESC
""")

---
## E9 · ORDER BY + LIMIT (Top-N)
**Task:** Return the **top 3 highest-paid** employees: name, department name, and salary.

In [ ]:
# ── Your answer ────────────────────────────────────────────────
q("""

""")

In [ ]:
# ── Answer E9 ──────────────────────────────────────────────────
q("""
    SELECT   e.name, d.dept_name, e.salary
    FROM     employees e
    JOIN     departments d ON e.dept_id = d.dept_id
    ORDER BY e.salary DESC
    LIMIT 3
""")

---
## E10 · UPDATE + SELECT
**Task:** Give all **Engineering** employees a **10% raise**, then display the updated Engineering salaries.

In [ ]:
# ── Your answer ────────────────────────────────────────────────
# Run the UPDATE, then SELECT to verify

In [ ]:
# ── Answer E10 ─────────────────────────────────────────────────
print('── Before raise (Engineering) ──')
display(q("""
    SELECT e.name, e.salary
    FROM   employees e JOIN departments d ON e.dept_id = d.dept_id
    WHERE  d.dept_name = 'Engineering'
"""))

# UPDATE
conn.execute("""
    UPDATE employees
    SET    salary = salary * 1.10
    WHERE  dept_id = (SELECT dept_id FROM departments WHERE dept_name = 'Engineering')
""")
conn.commit()

print('\n── After 10% raise ──')
display(q("""
    SELECT e.name, e.salary
    FROM   employees e JOIN departments d ON e.dept_id = d.dept_id
    WHERE  d.dept_name = 'Engineering'
"""))

---
## Bonus — pandas + SQL

**Task:** Load the employees table into a pandas DataFrame, then:
1. Find the department with the highest total salary budget
2. Export the result to a CSV file

In [ ]:
# ── Your answer ────────────────────────────────────────────────

In [ ]:
# ── Answer Bonus ───────────────────────────────────────────────
import pandas as pd

# 1. Load via pd.read_sql
df = pd.read_sql("""
    SELECT d.dept_name,
           COUNT(e.emp_id)          AS headcount,
           SUM(e.salary)            AS total_budget,
           ROUND(AVG(e.salary), 0)  AS avg_salary
    FROM   departments d
    LEFT JOIN employees e ON d.dept_id = e.dept_id
    GROUP BY d.dept_id, d.dept_name
    ORDER BY total_budget DESC
""", conn)

display(df)

# 2. Highest budget department
top_dept = df.loc[df['total_budget'].idxmax(), 'dept_name']
print(f'\nHighest salary budget: {top_dept}')

# 3. Export to CSV
df.to_csv('dept_salary_summary.csv', index=False)
print('Saved: dept_salary_summary.csv')